In [ ]:
import pandas as pd
import subprocess
import os

In [ ]:
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.','-')

%env USER_NAME={USER_NAME}

# SAIGE GRM separated by group

In [ ]:
cur_dir = '/home/jupyter/workspaces/genomicsofinfectiousdiseasesusceptibility/meta'

## Make parameter file for each ancestry

In [ ]:
# Initialize an Empty DataFrame
param = []

my_bucket = os.getenv('WORKSPACE_BUCKET')

for anc in ["eur", "afr", "eas", "amr", "oth", "mid", "sas"]:
    param.append({'--input-recursive bfile' : f"{my_bucket}/data/meta/plink_array/qc_{anc}",
                  '--output-recursive outpath' : f"{my_bucket}/data/meta/saige_grm/",
                  '--env ancestry' : anc})

param = pd.DataFrame(param)
print(param)

In [ ]:
PARAMETER_FILENAME = f'{cur_dir}/saige_grm_bygroupPARAM.tsv'

param.to_csv(PARAMETER_FILENAME, sep = '\t', index = False)

%env PARAMETER_FILENAME={PARAMETER_FILENAME}

## Script to compute GRM

In [ ]:
%%writefile script.sh

#!/bin/bash
    
# This script makes SAIGE GRM

set -o errexit
set -o nounset

createSparseGRM.R \
    --plinkFile="${bfile}/plinkarray_qc_${ancestry}" \
    --nThreads=15 \
    --memoryChunk=2 \
    --numRandomMarkerforSparseKin=5000 \
    --relatednessCutoff=0.125 \
    --minMAFforGRM=0.05 \
    --maxMissingRateforGRM=0.05 \
    --outputPrefix="${outpath}/${ancestry}"

In [ ]:
!gsutil mv script.sh ${WORKSPACE_BUCKET}/data/scripts/bash/saige_grm/saige_grm_bygroup.sh

In [ ]:
!gsutil cat ${WORKSPACE_BUCKET}/data/scripts/bash/saige_grm/saige_grm_bygroup.sh

## Submit bsub job

In [ ]:
#!gsutil rm -r ${WORKSPACE_BUCKET}/data/logs/saige-grm-bygroup

In [ ]:
%%bash --out job_id

source ~/aou_dsub.bash

aou_dsub \
    --image wzhou88/saige:1.3.6 \
    --min-ram 32 \
    --min-cores 16 \
    --logging "${WORKSPACE_BUCKET}/data/logs/{job-name}/{job-id}-{task-id}-{task-attempt}.log" \
    --tasks "${PARAMETER_FILENAME}" \
    --script "${WORKSPACE_BUCKET}/data/scripts/bash/saige_grm/saige_grm_bygroup.sh"

In [ ]:
%env JOB_ID={job_id}

In [ ]:
%%bash

dstat \
    --provider google-cls-v2 \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --users "${USER_NAME}" \
    --jobs "${JOB_ID}" \
    --status '*'